
*  DSC 530
*  Week 5 & 6 Assignment
*  Saebastion Cole

# Hands-on data analysis with Pandas

In [381]:
# Before I do anything I am starting with the imports.
# I am opting to stick with pandas with this assignment as pandas is the most widely used and probably the best to learn.

import numpy as np
import pandas as pd

# I will also load the data from the csv here.
earthquakes = pd.read_csv("data/earthquakes.csv")
faang = pd.read_csv("data/faang.csv")

# Chapter 4 Exercises

## Exercise 1.

With the earthquakes.csv file, select all the earthquakes in Japan with a magnitude of 4.9 or greater using the mb magnitude type.

To accomplish this I am going to just filter out the data based on the requirements.

In [382]:
# Create new dataframe from the filtered out ones

japan_earthquakes = earthquakes[earthquakes["place"].str.contains("Japan") &
 (earthquakes["mag"] >= 4.9) & (earthquakes["magType"] == "mb")]

japan_earthquakes.head()

,mag,magType,time,place,tsunami,parsed_place
1563,4.9,mb,1538977532250,"293km ESE of Iwo Jima, Japan",0,Japan
2576,5.4,mb,1538697528010,"37km E of Tomakomai, Japan",0,Japan
3072,4.9,mb,1538579732490,"15km ENE of Hasaki, Japan",0,Japan
3632,4.9,mb,1538450871260,"53km ESE of Hitachi, Japan",0,Japan


## Exercise 2.

Create bins for each full number of earthquake magnitude (for instance, the first bin is (0,1], the second is (1,2], and so on) with the ml magnitude type and count how many are in each bin.

In [383]:
# First filter out the data for ml only type.

ml_earthquakes = earthquakes[earthquakes["magType"] == "ml"]

# Next we will use the cut() function to create the bins for the magnitude ranges.
# A quick google search tells me that ml refers to the richter scale with a max of 7.0 so I will make the bins from 0 - 7.
# Also I found the range function helpfully generates the intervals for me simplifying the code.

earthquakes_bins = pd.cut(ml_earthquakes["mag"], bins=range(0, 7))

earthquakes_bins.head()

0    (1, 2]
1    (1, 2]
2    (3, 4]
3    (0, 1]
6    (1, 2]
Name: mag, dtype: category
Categories (6, interval[int64, right]): [(0, 1] < (1, 2] < (2, 3] < (3, 4] < (4, 5] < (5, 6]]

Now that I have the dataset binned I will calculate the counts.

In [384]:
# value_counts() will give me the counts of each bin.
earthquakes_bins.value_counts()

mag
(1, 2]    3105
(0, 1]    2207
(2, 3]     862
(3, 4]     122
(4, 5]       2
(5, 6]       1
Name: count, dtype: int64

## Exercise 4.

Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum magnitude that was observed for each combination. Put the magnitude type along the columns.

In [385]:
# Pandas has a built in function for this named crosstab()
crosstab_earthquakes = pd.crosstab(
    index=earthquakes["tsunami"], 
    columns=earthquakes["magType"],
    values=earthquakes["mag"],
    aggfunc="max"
    )
crosstab_earthquakes.head()

magType,mb,mb_lg,md,mh,ml,ms_20,mw,mwb,mwr,mww
tsunami,,,,,,,,,,
0,5.6,3.5,4.11,1.1,4.2,NaN,3.83,5.8,4.8,6.0
1,6.1,NaN,NaN,NaN,5.1,5.7,4.41,NaN,NaN,7.5


## Exercise 6.

Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.

In [386]:
# Pandas pivot_table will let me transform the data for this.
# OHLC means open, high, low, close

pivot_faang = faang.pivot_table(
    index="ticker", 
    values=["open", "high", "low", "close", "volume"],
    aggfunc="mean"
    )
pivot_faang.head()

,close,high,low,open,volume
ticker,,,,,
AAPL,47.263357,47.748526,46.795877,47.277859,1.360803e+08
AMZN,1641.726176,1662.839839,1619.840519,1644.072709,5.648994e+06
FB,171.510956,173.613347,169.303148,171.472948,2.765860e+07
GOOG,1113.225134,1125.777606,1101.001658,1113.554101,1.741965e+06
NFLX,319.290319,325.219322,313.187330,319.620558,1.146962e+07


## Exercise 7.

Calculate the z-scores for each numeric column of Amazon’s data (ticker is AMZN) in Q4 2018 using apply().



In [387]:
# First I get only amazon data
amzn_data = faang[(faang["ticker"] == "AMZN") & (faang["date"] >= "2018-10-01") & (faang["date"] <= "2018-12-31")]

# I can only use the numeric columns for zscore
numeric = amzn_data[["open", "high", "low", "close", "volume"]]
zscore = (numeric - numeric.mean()) / numeric.std()
zscore.head()

,open,high,low,close,volume
690,2.337813,2.368006,2.502113,2.385848,-1.630411
691,2.190795,2.227302,2.247433,2.155037,-0.861879
692,2.068570,2.058955,2.139987,2.025489,-0.920345
693,1.850048,1.819474,1.781561,1.722816,-0.126582
694,1.642819,1.628173,1.554416,1.584748,-0.298771


## Exercise 10.

The European Centre for Disease Prevention and Control (ECDC) provides an open dataset on COVID-19 cases called daily number of new reported cases of COVID-19 by country worldwide. This dataset is updated daily, but we will use a snapshot that contains data through September 18, 2020. Complete the following tasks to practice the skills you've learned up to this point in the book:

- Prepare the data
    - Read in the data in the covid19_cases.csv file.

    - Create a date column by parsing the dateRep column into a datetime.
    - Set the date column as the index.
    - Use the replace() method to update all occurrences of United_States_of_America and United_Kingdom to USA and UK, respectively.
    - Sort the index.

- For the five countries with the most cases (cumulative), find the day with the largest number of cases.
- Find the 7-day average change in COVID-19 cases for the last week in the data for the five countries with the most cases.
- Find the first date that each country other than China had cases.
- Rank the countries by cumulative cases using percentiles.

In [388]:
# Load the data
 
covid_data = pd.read_csv("data/covid_cases.csv")

# Create the new column for data
covid_data["date"] = pd.to_datetime(covid_data["dateRep"], format="%d/%m/%Y")
covid_data.head()

,dateRep,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000,date
0,14/12/2020,14,12,2020,746,6,Afghanistan,AF,AFG,38041757.0,Asia,9.013779,2020-12-14
1,13/12/2020,13,12,2020,298,9,Afghanistan,AF,AFG,38041757.0,Asia,7.052776,2020-12-13
2,12/12/2020,12,12,2020,113,11,Afghanistan,AF,AFG,38041757.0,Asia,6.868768,2020-12-12
3,11/12/2020,11,12,2020,63,10,Afghanistan,AF,AFG,38041757.0,Asia,7.134266,2020-12-11
4,10/12/2020,10,12,2020,202,16,Afghanistan,AF,AFG,38041757.0,Asia,6.968658,2020-12-10


In [389]:

# Set the index
covid_data = covid_data.set_index("date")



covid_data["countriesAndTerritories"].replace({"United_States_of_America": "USA", "United_Kingdom": "UK"})
covid_data.head()

,dateRep,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
date,,,,,,,,,,,,
2020-12-14,14/12/2020,14,12,2020,746,6,Afghanistan,AF,AFG,38041757.0,Asia,9.013779
2020-12-13,13/12/2020,13,12,2020,298,9,Afghanistan,AF,AFG,38041757.0,Asia,7.052776
2020-12-12,12/12/2020,12,12,2020,113,11,Afghanistan,AF,AFG,38041757.0,Asia,6.868768
2020-12-11,11/12/2020,11,12,2020,63,10,Afghanistan,AF,AFG,38041757.0,Asia,7.134266
2020-12-10,10/12/2020,10,12,2020,202,16,Afghanistan,AF,AFG,38041757.0,Asia,6.968658


In [390]:
# Sort it
covid_data = covid_data.sort_index()
covid_data.head()

,dateRep,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
date,,,,,,,,,,,,
2019-12-31,31/12/2019,31,12,2019,0,0,Iraq,IQ,IRQ,39309789.0,Asia,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Netherlands,NL,NLD,17282163.0,Europe,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Israel,IL,ISR,8519373.0,Asia,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,New_Zealand,NZ,NZL,4783062.0,Oceania,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Dominican_Republic,DO,DOM,10738957.0,America,NaN


In [391]:
# Get the 5 highest case countries
top5 = covid_data.groupby("countriesAndTerritories")["cases"].sum().nlargest(5)
top5.head()

countriesAndTerritories
United_States_of_America    16256754
India                        9884100
Brazil                       6901952
Russia                       2653928
France                       2376852
Name: cases, dtype: int64

In [392]:
# Get the most cases in a given day

most_cases = covid_data.groupby("date")["cases"].sum().nlargest(1)
most_cases.head()

date
2020-12-11    693352
Name: cases, dtype: int64

In [393]:
# Get the 7 day average change

pivot = covid_data.pivot_table(
      index="date",
      columns="countriesAndTerritories",
      values="cases",
      aggfunc="sum"
  ).sort_index()

# This gets the differences between the mean and the value in a range of 7 days.
# Also only picks the ones who were in the top 5 countries
result = pivot[top5.index].diff().rolling(7).mean().tail(7)
result

countriesAndTerritories,United_States_of_America,India,Brazil,Russia,France
date,,,,,
2020-12-08,5633.000000,-650.142857,-109.571429,257.714286,-84.857143
2020-12-09,5274.714286,-646.285714,25.571429,-43.571429,804.285714
2020-12-10,2387.714286,-575.714286,512.857143,120.714286,75.857143
2020-12-11,1419.000000,-1028.142857,416.142857,-31.142857,150.571429
2020-12-12,386.142857,-949.428571,1077.714286,168.857143,312.142857
2020-12-13,583.428571,-822.428571,98.714286,-92.142857,146.285714
2020-12-14,2327.285714,-844.285714,-648.285714,-137.000000,73.000000


In [394]:
# Get all of the first records of countries

first_date = covid_data[covid_data.index == covid_data.index.min()]

# Filter out China
first_date = first_date[first_date["countriesAndTerritories"] != "China"]

first_date.head()

,dateRep,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
date,,,,,,,,,,,,
2019-12-31,31/12/2019,31,12,2019,0,0,Iraq,IQ,IRQ,39309789.0,Asia,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Netherlands,NL,NLD,17282163.0,Europe,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Israel,IL,ISR,8519373.0,Asia,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,New_Zealand,NZ,NZL,4783062.0,Oceania,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Dominican_Republic,DO,DOM,10738957.0,America,NaN


In [ ]:
# Rank cumulitatively cases by percentiles

cumulative = covid_data.groupby("countriesAndTerritories")["cases"].sum()

# This function actually puts them into their percentiles and then sorts them in descending order.
percentiles = cumulative.rank(pct=True).sort_values(ascending=False)

percentiles.head()

countriesAndTerritories
United_States_of_America    1.000000
India                       0.995327
Brazil                      0.990654
Russia                      0.985981
France                      0.981308
Name: cases, dtype: float64